## LiTS checkpoint evaluation (best vs latest)

This notebook **only evaluates** frozen checkpoints on your CSV splits. It follows common medical-segmentation practice:

- **Overlap**: Dice (DSC) and IoU (Jaccard), plus precision/recall for the rare tumor class ([BMC Research Notes evaluation guideline](https://bmcresnotes.biomedcentral.com/article/10.1186/s13104-022-06096-y)).
- **Decision / uncertainty**: **Threshold calibration on the validation set only**, then **one locked evaluation on the test set** with those thresholds (no test-time tuning; aligns with discrimination vs leakage discussion in [Metrics Reloaded, *Nature Methods* 2023](https://www.nature.com/articles/s41592-023-02151-z)).
- **Statistics**: **Study-level** resampling for bootstrap 95% CIs (slices from the same patient are correlated).
- **Clinical plausibility**: **Tumor-outside-liver rate** (fraction of predicted tumor pixels not covered by GT liver).
- **Reporting**: CSV + JSON summaries, **comparison bar charts**, and **qualitative PNG panels** (GT vs both checkpoints).

> Default `num_workers=0` avoids Kaggle notebook multiprocessing shutdown bugs.


In [ ]:
# Cell 1 — Imports, config, paths
import os
import json
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import torch.multiprocessing as mp

warnings.filterwarnings("ignore", category=UserWarning)

CFG = {
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "image_size": 256,
    "batch_size": 16,
    "num_workers": 0,
    "pin_memory": False,
    "save_dir": "results",
    "model_dir": "models",
    "figures_subdir": "eval_figures",
    "threshold_grid_size": 5,
    "bootstrap_rounds": 2000,
    "bootstrap_seed": 42,
    "median_probe_rows": 400,
    "viz_num_studies_with_tumor": 4,
    "viz_num_studies_tumor_free": 2,
    "viz_dpi": 140,
}

# --- Paste Kaggle / local paths here ---
DATA_ROOT_OVERRIDE = ""
CSV_ROOT_OVERRIDE = ""
VAL_CSV_OVERRIDE = ""
TEST_CSV_OVERRIDE = ""
MODEL_DIR_OVERRIDE = ""
BEST_CKPT_OVERRIDE = ""
LATEST_CKPT_OVERRIDE = ""

KAGGLE_DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/andrewmvd/lits-png/dataset_6"),
    Path("/kaggle/input/datasets/andrewmvd/lits-png/dataset_6/dataset_6"),
]
LOCAL_DATA_ROOT = Path("data/dataset_6")
CSV_ROOT = Path("data")


def pick_existing_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None


kaggle_root = pick_existing_path(KAGGLE_DATA_ROOT_CANDIDATES)
PATHS = {
    "data_root": kaggle_root if kaggle_root is not None else LOCAL_DATA_ROOT,
    "val_csv": CSV_ROOT / "lits_val.csv",
    "test_csv": CSV_ROOT / "lits_test.csv",
}

if DATA_ROOT_OVERRIDE.strip():
    PATHS["data_root"] = Path(DATA_ROOT_OVERRIDE.strip())
if CSV_ROOT_OVERRIDE.strip():
    croot = Path(CSV_ROOT_OVERRIDE.strip())
    PATHS["val_csv"] = croot / "lits_val.csv"
    PATHS["test_csv"] = croot / "lits_test.csv"
if VAL_CSV_OVERRIDE.strip():
    PATHS["val_csv"] = Path(VAL_CSV_OVERRIDE.strip())
if TEST_CSV_OVERRIDE.strip():
    PATHS["test_csv"] = Path(TEST_CSV_OVERRIDE.strip())
if MODEL_DIR_OVERRIDE.strip():
    CFG["model_dir"] = MODEL_DIR_OVERRIDE.strip()

CKPT_PATHS = {
    "best": Path(BEST_CKPT_OVERRIDE.strip()) if BEST_CKPT_OVERRIDE.strip() else Path(CFG["model_dir"]) / "best_model_tumor_dice.pth",
    "latest": Path(LATEST_CKPT_OVERRIDE.strip()) if LATEST_CKPT_OVERRIDE.strip() else Path(CFG["model_dir"]) / "latest_model.pth",
}


def resolve_data_root_with_csv(root_path: Path, csv_path: Path) -> Path:
    if not csv_path.exists():
        return root_path
    df_tmp = pd.read_csv(csv_path, nrows=1)
    if len(df_tmp) == 0:
        return root_path
    sample_name = Path(str(df_tmp.iloc[0]["filepath"])).name
    candidates = [
        root_path,
        root_path / "dataset_6",
        root_path.parent / "dataset_6",
        root_path.parent.parent / "dataset_6",
    ]
    for cand in candidates:
        if (cand / sample_name).exists():
            return cand
    return root_path


PATHS["data_root"] = resolve_data_root_with_csv(PATHS["data_root"], PATHS["val_csv"])

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

Path(CFG["save_dir"]).mkdir(parents=True, exist_ok=True)
(Path(CFG["save_dir"]) / CFG["figures_subdir"]).mkdir(parents=True, exist_ok=True)

print(f"Device: {CFG['device']}")
print("Using paths:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")
print("Checkpoint paths:")
for k, v in CKPT_PATHS.items():
    print(f"  {k}: {v}")

assert PATHS["data_root"].exists(), f"Missing data_root: {PATHS['data_root']}"
assert PATHS["val_csv"].exists(), f"Missing val_csv: {PATHS['val_csv']}"
assert PATHS["test_csv"].exists(), f"Missing test_csv: {PATHS['test_csv']}"
assert CKPT_PATHS["best"].exists(), f"Missing best checkpoint: {CKPT_PATHS['best']}"
assert CKPT_PATHS["latest"].exists(), f"Missing latest checkpoint: {CKPT_PATHS['latest']}"

if CKPT_PATHS["best"].resolve() == CKPT_PATHS["latest"].resolve():
    print("Warning: best and latest point to the same file — comparison is trivial unless you override paths.")


In [ ]:
# Cell 2 — Preprocessing + dataset + U-Net (match training notebook)
class CTPreprocessor:
    def __init__(self, image_size=256, use_clahe=True, apply_median=False):
        self.image_size = image_size
        self.use_clahe = use_clahe
        self.apply_median = apply_median
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __call__(self, image_u8: np.ndarray) -> np.ndarray:
        x = image_u8.copy()
        lo, hi = np.percentile(x, 0.5), np.percentile(x, 99.5)
        if hi > lo:
            x = np.clip(x, lo, hi)
            x = ((x - lo) / (hi - lo + 1e-8) * 255.0).astype(np.uint8)
        if self.apply_median:
            x = cv2.medianBlur(x, 3)
        if self.use_clahe:
            x = self.clahe.apply(x)
        x = cv2.resize(x, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        return x


def resolve_paths(row: pd.Series, root: Path):
    return (
        root / Path(str(row["filepath"])).name,
        root / Path(str(row["liver_maskpath"])).name,
        root / Path(str(row["tumor_maskpath"])).name,
    )


class LITSDatasetStrict(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path, image_size=256, use_clahe=True, apply_median=False):
        self.df = df.reset_index(drop=True).copy()
        self.root_dir = Path(root_dir)
        self.preprocess = CTPreprocessor(image_size=image_size, use_clahe=use_clahe, apply_median=apply_median)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path, liv_path, tum_path = resolve_paths(row, self.root_dir)
        study = int(row["study_number"])

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        liv = cv2.imread(str(liv_path), cv2.IMREAD_GRAYSCALE)
        tum = cv2.imread(str(tum_path), cv2.IMREAD_GRAYSCALE)
        if img is None or liv is None or tum is None:
            exists = img_path.exists()
            raise ValueError(f"Unreadable triple at idx={idx} | img={img_path} | exists={exists}")

        img = self.preprocess(img)
        liv = cv2.resize(liv, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        tum = cv2.resize(tum, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        liv = (liv > 0).astype(np.float32)
        tum = (tum > 0).astype(np.float32)

        x = torch.from_numpy(img.astype(np.float32) / 255.0).unsqueeze(0)
        y = torch.from_numpy(np.stack([liv, tum], axis=0).astype(np.float32))
        return x, y, study


def collate_study(batch):
    xs = torch.stack([b[0] for b in batch], dim=0)
    ys = torch.stack([b[1] for b in batch], dim=0)
    st = torch.tensor([b[2] for b in batch], dtype=torch.long)
    return xs, ys, st


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=2, feat=(64, 128, 256, 512)):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()

        ch = in_ch
        for f in feat:
            self.downs.append(DoubleConv(ch, f))
            ch = f
        self.bottleneck = DoubleConv(feat[-1], feat[-1] * 2)

        rev = list(reversed(feat))
        up_in = feat[-1] * 2
        for f in rev:
            self.ups.append(nn.ConvTranspose2d(up_in, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f * 2, f))
            up_in = f

        self.head = nn.Conv2d(feat[0], out_ch, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i // 2]
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = self.ups[i + 1](torch.cat([skip, x], dim=1))

        return self.head(x)


def load_checkpoint_state_dict(path: Path, device: str):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        return ckpt["model_state_dict"], ckpt
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        return ckpt["state_dict"], ckpt
    return ckpt, {"epoch": -1}


def build_model_from_ckpt(path: Path, device: str):
    sd, meta = load_checkpoint_state_dict(path, device)
    model = UNet(in_ch=1, out_ch=2).to(device)
    model.load_state_dict(sd, strict=True)
    model.eval()
    return model, meta


In [ ]:
# Cell 3 — Auto median policy (match training notebook) + DataLoaders
val_df = pd.read_csv(PATHS["val_csv"])
test_df = pd.read_csv(PATHS["test_csv"])

base_cols = {"filepath", "liver_maskpath", "tumor_maskpath", "study_number"}
for name, df in [("val", val_df), ("test", test_df)]:
    miss = base_cols - set(df.columns)
    assert not miss, f"{name} CSV missing columns: {miss}"
if "tumor_mask_empty" not in test_df.columns:
    raise ValueError("test CSV must include tumor_mask_empty for stratified qualitative sampling")

use_clahe_policy = True


def estimate_mean_tumor_fraction(df: pd.DataFrame, root: Path, max_rows: int) -> float:
    rng = np.random.default_rng(0)
    n = min(len(df), max_rows)
    if n <= 0:
        return 0.0
    idxs = rng.choice(len(df), size=n, replace=False)
    fracs = []
    for i in idxs:
        row = df.iloc[i]
        _, _, tum_path = resolve_paths(row, root)
        tum = cv2.imread(str(tum_path), cv2.IMREAD_GRAYSCALE)
        if tum is None:
            continue
        tum = cv2.resize(tum, (CFG["image_size"], CFG["image_size"]), interpolation=cv2.INTER_NEAREST)
        fracs.append(float((tum > 0).mean()))
    return float(np.mean(fracs)) if fracs else 0.0


mean_tumor_px = estimate_mean_tumor_fraction(val_df, PATHS["data_root"], CFG["median_probe_rows"])
apply_median_policy = mean_tumor_px < 0.003
print(f"Preprocess | CLAHE={use_clahe_policy} | median={apply_median_policy} | mean tumor fg (sampled)={mean_tumor_px:.6f}")

val_ds = LITSDatasetStrict(
    val_df, PATHS["data_root"], image_size=CFG["image_size"], use_clahe=use_clahe_policy, apply_median=apply_median_policy
)
test_ds = LITSDatasetStrict(
    test_df, PATHS["data_root"], image_size=CFG["image_size"], use_clahe=use_clahe_policy, apply_median=apply_median_policy
)

loader_kw = dict(num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"], collate_fn=collate_study)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, **loader_kw)
test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False, **loader_kw)
print(f"Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


In [ ]:
# Cell 4 — Metrics, study-level bootstrap, evaluation
SMOOTH = 1e-6


def bootstrap_ci_study(study_scores: dict, n_boot: int, alpha: float, seed: int):
    studies = np.array(sorted(study_scores.keys()), dtype=np.int64)
    if studies.size == 0:
        return float("nan"), float("nan"), float("nan")
    vals = np.array([study_scores[s] for s in studies], dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    means = []
    n = vals.size
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        means.append(float(np.mean(vals[idx])))
    means = np.asarray(means)
    lo = float(np.percentile(means, 100 * (alpha / 2)))
    hi = float(np.percentile(means, 100 * (1 - alpha / 2)))
    return float(np.mean(vals)), lo, hi


def dice_per_slice(pred_ch, gt_ch):
    p = pred_ch.reshape(-1)
    t = gt_ch.reshape(-1)
    inter = float((p * t).sum().item())
    denom = float(p.sum().item() + t.sum().item())
    if denom <= 0:
        return float("nan")
    return (2 * inter + SMOOTH) / (denom + SMOOTH)


def global_stats_from_conf(tp, fp, fn, tn):
    dice = (2 * tp + SMOOTH) / (2 * tp + fp + fn + SMOOTH)
    iou = (tp + SMOOTH) / (tp + fp + fn + SMOOTH)
    precision = (tp + SMOOTH) / (tp + fp + SMOOTH)
    recall = (tp + SMOOTH) / (tp + fn + SMOOTH)
    f1 = (2 * precision * recall) / (precision + recall + SMOOTH)
    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": float(tp),
        "fp": float(fp),
        "fn": float(fn),
        "tn": float(tn),
    }


@torch.no_grad()
def eval_checkpoint(model, loader, th_liver=0.5, th_tumor=0.5):
    model.eval()
    tp = np.zeros(2, dtype=np.float64)
    fp = np.zeros(2, dtype=np.float64)
    fn = np.zeros(2, dtype=np.float64)
    tn = np.zeros(2, dtype=np.float64)
    tum_outside = 0.0
    tum_pred_px = 0.0

    liver_by_study = {}
    tumor_by_study = {}

    use_amp = CFG["device"] == "cuda"
    for x, y, studies in tqdm(loader, leave=False):
        x = x.to(CFG["device"], non_blocking=True)
        y = y.to(CFG["device"], non_blocking=True).float()
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(x)
        probs = torch.sigmoid(logits)
        pred_l = (probs[:, 0] > th_liver).float()
        pred_t = (probs[:, 1] > th_tumor).float()
        pred = torch.stack([pred_l, pred_t], dim=1)
        gt_l = y[:, 0]

        tum_outside += float(((pred_t > 0) & (gt_l < 0.5)).sum().item())
        tum_pred_px += float((pred_t > 0).sum().item())

        for ch in [0, 1]:
            p = pred[:, ch].reshape(-1).detach().cpu().numpy()
            t = y[:, ch].reshape(-1).detach().cpu().numpy()
            tp[ch] += float((p * t).sum())
            fp[ch] += float((p * (1 - t)).sum())
            fn[ch] += float(((1 - p) * t).sum())
            tn[ch] += float(((1 - p) * (1 - t)).sum())

        bsz = pred.shape[0]
        for bi in range(bsz):
            st = int(studies[bi].item())
            dl = dice_per_slice(pred[bi, 0], y[bi, 0])
            dt = dice_per_slice(pred[bi, 1], y[bi, 1])
            liver_by_study.setdefault(st, []).append(dl)
            tumor_by_study.setdefault(st, []).append(dt)

    liver_study_mean = {s: float(np.nanmean(v)) for s, v in liver_by_study.items()}
    tumor_study_mean = {s: float(np.nanmean(v)) for s, v in tumor_by_study.items()}

    liver_mean, liver_lo, liver_hi = bootstrap_ci_study(
        liver_study_mean, CFG["bootstrap_rounds"], 0.05, CFG["bootstrap_seed"]
    )
    tumor_mean, tumor_lo, tumor_hi = bootstrap_ci_study(
        tumor_study_mean, CFG["bootstrap_rounds"], 0.05, CFG["bootstrap_seed"] + 1
    )

    liver = global_stats_from_conf(tp[0], fp[0], fn[0], tn[0])
    tumor = global_stats_from_conf(tp[1], fp[1], fn[1], tn[1])
    tumor_fp_rate = tumor["fp"] / max(1.0, (tumor["fp"] + tumor["tn"]))
    out_rate = tum_outside / max(1.0, tum_pred_px)

    return {
        "liver": liver,
        "tumor": tumor,
        "mean_dice": float((liver["dice"] + tumor["dice"]) / 2),
        "tumor_fp_rate": float(tumor_fp_rate),
        "tumor_outside_liver_rate": float(out_rate),
        "liver_dice_ci95_study": [liver_mean, liver_lo, liver_hi],
        "tumor_dice_ci95_study": [tumor_mean, tumor_lo, tumor_hi],
        "liver_by_study_mean": liver_study_mean,
        "tumor_by_study_mean": tumor_study_mean,
    }


def balanced_score_dict(out):
    return (
        0.50 * out["tumor"]["dice"]
        + 0.36 * out["liver"]["dice"]
        - 0.08 * out["tumor_fp_rate"]
        - 0.06 * out["tumor_outside_liver_rate"]
    )


def summarize_ckpt_for_json(ckpt_meta):
    if not isinstance(ckpt_meta, dict):
        return {}
    keys = [k for k in ("epoch", "val_clinical_score", "val_tumor_dice", "val_liver_dice") if k in ckpt_meta]
    return {k: ckpt_meta[k] for k in keys}


In [ ]:
# Cell 5 — Calibrate on val, lock test, export tables + lean JSON
ckpt_paths = CKPT_PATHS
threshold_grid = np.linspace(0.30, 0.70, CFG["threshold_grid_size"])
print(f"Threshold sweep: {len(threshold_grid)} x {len(threshold_grid)} = {len(threshold_grid)**2} val passes per checkpoint")

rows = []
per_study_rows = []
detailed_summaries = {}

for ckpt_name, ckpt_path in ckpt_paths.items():
    model, ckpt_meta = build_model_from_ckpt(ckpt_path, CFG["device"])

    best_val = None
    for th_t in threshold_grid:
        for th_l in threshold_grid:
            val_out = eval_checkpoint(model, val_loader, th_liver=float(th_l), th_tumor=float(th_t))
            score = balanced_score_dict(val_out)
            if best_val is None or score > best_val["balanced_score"]:
                best_val = {
                    "th_liver": float(th_l),
                    "th_tumor": float(th_t),
                    "balanced_score": float(score),
                    "val_out": val_out,
                }

    test_out = eval_checkpoint(
        model, test_loader, th_liver=best_val["th_liver"], th_tumor=best_val["th_tumor"]
    )

    detailed_summaries[ckpt_name] = {
        "th_liver": best_val["th_liver"],
        "th_tumor": best_val["th_tumor"],
        "val_balanced_score": best_val["balanced_score"],
        "test_balanced_score": float(balanced_score_dict(test_out)),
        "checkpoint_meta": summarize_ckpt_for_json(ckpt_meta),
        "val_liver_dice": best_val["val_out"]["liver"]["dice"],
        "val_tumor_dice": best_val["val_out"]["tumor"]["dice"],
        "test_liver_dice": test_out["liver"]["dice"],
        "test_tumor_dice": test_out["tumor"]["dice"],
        "test_tumor_outside_liver_rate": test_out["tumor_outside_liver_rate"],
        "test_tumor_fp_rate": test_out["tumor_fp_rate"],
    }

    for split_name, out in [("val", best_val["val_out"]), ("test", test_out)]:
        lmean, llo, lhi = out["liver_dice_ci95_study"]
        tmean, tlo, thi = out["tumor_dice_ci95_study"]
        rows.append({
            "checkpoint": ckpt_name,
            "split": split_name,
            "th_liver": best_val["th_liver"],
            "th_tumor": best_val["th_tumor"],
            "liver_dice_micro": out["liver"]["dice"],
            "tumor_dice_micro": out["tumor"]["dice"],
            "mean_dice_micro": out["mean_dice"],
            "tumor_fp_rate": out["tumor_fp_rate"],
            "tumor_outside_liver_rate": out["tumor_outside_liver_rate"],
            "liver_iou": out["liver"]["iou"],
            "tumor_iou": out["tumor"]["iou"],
            "tumor_precision": out["tumor"]["precision"],
            "tumor_recall": out["tumor"]["recall"],
            "liver_dice_ci95_study_mean": lmean,
            "liver_dice_ci95_study_lo": llo,
            "liver_dice_ci95_study_hi": lhi,
            "tumor_dice_ci95_study_mean": tmean,
            "tumor_dice_ci95_study_lo": tlo,
            "tumor_dice_ci95_study_hi": thi,
            "balanced_score": float(balanced_score_dict(out)),
        })

    for st, v in test_out["liver_by_study_mean"].items():
        per_study_rows.append({
            "checkpoint": ckpt_name,
            "study_number": st,
            "test_liver_dice_mean_slices": v,
            "test_tumor_dice_mean_slices": test_out["tumor_by_study_mean"].get(st, float("nan")),
            "th_liver": best_val["th_liver"],
            "th_tumor": best_val["th_tumor"],
        })

    del model
    if CFG["device"] == "cuda":
        torch.cuda.empty_cache()

compare_df = pd.DataFrame(rows)
fig_dir = Path(CFG["save_dir"]) / CFG["figures_subdir"]
out_csv = Path(CFG["save_dir"]) / "checkpoint_comparison.csv"
out_study = Path(CFG["save_dir"]) / "per_study_test_dice.csv"
out_json = Path(CFG["save_dir"]) / "checkpoint_recommendation.json"
out_detailed = Path(CFG["save_dir"]) / "checkpoint_eval_summary.json"

compare_df.to_csv(out_csv, index=False)
pd.DataFrame(per_study_rows).to_csv(out_study, index=False)

cmp_test = compare_df[compare_df["split"] == "test"].copy().sort_values("balanced_score", ascending=False)
best_row = cmp_test.iloc[0]
def _jsonify_row(row):
    out = {}
    for k, v in row.items():
        if hasattr(v, "item"):
            v = v.item()
        if isinstance(v, float) or isinstance(v, (np.floating,)):
            out[k] = float(v)
        elif isinstance(v, int) or isinstance(v, (np.integer,)):
            out[k] = int(v)
        else:
            out[k] = v
    return out


recommendation = {
    "recommended_checkpoint": str(best_row["checkpoint"]),
    "rationale": "Validation-only threshold search; test used once with locked thresholds. Score balances tumor/liver Dice and penalizes FP and tumor-outside-liver.",
    "thresholds_from_recommended_ckpt": {
        "liver": float(best_row["th_liver"]),
        "tumor": float(best_row["th_tumor"]),
    },
    "test_row": _jsonify_row(best_row),
}
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(recommendation, f, indent=2)
with open(out_detailed, "w", encoding="utf-8") as f:
    json.dump(detailed_summaries, f, indent=2)

from datetime import datetime, timezone

out_log = Path(CFG["save_dir"]) / "eval_analysis_log.txt"
log_lines = []
log_lines.append("=" * 72)
log_lines.append("LiTS CHECKPOINT EVALUATION — ANALYSIS LOG")
log_lines.append("=" * 72)
log_lines.append(f"Generated (UTC): {datetime.now(timezone.utc).isoformat()}")
log_lines.append("")
log_lines.append("1) LEARNING SETUP (supervised segmentation)")
log_lines.append("-" * 72)
log_lines.append(
    "This is fully SUPERVISED learning: each sample is a CT slice (input) paired with "
    "dense PIXEL labels from expert-drawn PNG masks (liver + lesion/tumor). The model is "
    "trained to match those masks (loss on logits vs ground truth)."
)
log_lines.append("")
log_lines.append("How labels are used:")
log_lines.append(
    "  • TRAIN / VAL / TEST (training notebook): the loss and Dice-style metrics use the "
    "liver and tumor mask images loaded from CSV paths liver_maskpath and tumor_maskpath. "
    "That is the ground truth for every pixel."
)
log_lines.append(
    "  • CSV column tumor_mask_empty: row-level metadata (whether the lesion mask is empty "
    "on that slice). It is NOT the segmentation target. In this eval notebook it is only "
    "used to stratify qualitative PNG examples (studies with vs without tumor on slices)."
)
log_lines.append(
    "  • This notebook: no gradient updates; metrics compare thresholded predictions to the "
    "same mask files on validation and test."
)
log_lines.append("")
log_lines.append("2) RUN CONFIGURATION")
log_lines.append("-" * 72)
log_lines.append(f"device: {CFG['device']}")
log_lines.append(f"image_size: {CFG['image_size']} | batch_size: {CFG['batch_size']}")
log_lines.append(
    f"threshold_grid: {threshold_grid[0]:.2f} … {threshold_grid[-1]:.2f} ({len(threshold_grid)} steps) "
    f"=> {len(threshold_grid)**2} val sweeps per checkpoint"
)
log_lines.append(f"bootstrap_rounds (study-level Dice CI): {CFG['bootstrap_rounds']}")
try:
    log_lines.append(
        f"preprocess: CLAHE={use_clahe_policy} | median_blur={apply_median_policy} | "
        f"mean_tumor_fg_sampled={mean_tumor_px:.6f}"
    )
except NameError:
    log_lines.append("preprocess: (see cell 3 — mean_tumor_px / apply_median_policy)")
log_lines.append(f"data_root: {PATHS['data_root']}")
log_lines.append(f"val_csv:   {PATHS['val_csv']}  (rows={len(val_df)})")
log_lines.append(f"test_csv:  {PATHS['test_csv']} (rows={len(test_df)})")
for k, pth in CKPT_PATHS.items():
    log_lines.append(f"checkpoint[{k}]: {pth}")
log_lines.append(f"figures_dir: {fig_dir}")
log_lines.append("")
log_lines.append("3) METRIC TABLE (all rows)")
log_lines.append("-" * 72)
log_lines.append(compare_df.to_string(index=False))
log_lines.append("")
log_lines.append("4) PER-CHECKPOINT SUMMARY (post calibration)")
log_lines.append("-" * 72)
for name, summ in detailed_summaries.items():
    log_lines.append(f"[{name}]")
    for k2, v2 in summ.items():
        log_lines.append(f"  {k2}: {v2}")
    log_lines.append("")
log_lines.append("5) RECOMMENDATION")
log_lines.append("-" * 72)
log_lines.append(f"Recommended checkpoint: {recommendation['recommended_checkpoint']}")
log_lines.append(f"Rationale: {recommendation['rationale']}")
log_lines.append(
    "Thresholds (from recommended test row): "
    f"liver={recommendation['thresholds_from_recommended_ckpt']['liver']}, "
    f"tumor={recommendation['thresholds_from_recommended_ckpt']['tumor']}"
)
log_lines.append("")
log_lines.append("6) OUTPUT FILES")
log_lines.append("-" * 72)
log_lines.append(str(out_csv))
log_lines.append(str(out_study))
log_lines.append(str(out_json))
log_lines.append(str(out_detailed))
log_lines.append(str(out_log))
log_lines.append(str(fig_dir) + "/*.png")
log_lines.append("")
log_lines.append("END OF LOG")

out_log.write_text(chr(10).join(log_lines), encoding="utf-8")

print(compare_df.to_string(index=False))
print(f"\nSaved: {out_csv}\nSaved: {out_study}\nSaved: {out_json}\nSaved: {out_detailed}\nSaved: {out_log}")
print(f"\nRecommended checkpoint: {recommendation['recommended_checkpoint']}")


In [ ]:
# Cell 6 — Figures: metric comparison + qualitative side-by-side PNGs
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

test_cmp = compare_df[compare_df["split"] == "test"].set_index("checkpoint")

metrics = [
    ("liver_dice_micro", "Liver Dice (micro)"),
    ("tumor_dice_micro", "Tumor Dice (micro)"),
    ("tumor_outside_liver_rate", "Tumor outside liver rate"),
    ("tumor_fp_rate", "Tumor FP rate"),
    ("balanced_score", "Balanced score"),
]

fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 4), constrained_layout=True)
ck_names = list(test_cmp.index)
x = np.arange(len(ck_names))
for ax, (col, title) in zip(axes, metrics):
    vals = [float(test_cmp.loc[c, col]) for c in ck_names]
    ax.bar(x, vals, color=["#2ecc71", "#3498db"][: len(ck_names)])
    ax.set_xticks(x)
    ax.set_xticklabels(ck_names)
    ax.set_title(title)
    ax.grid(True, axis="y", alpha=0.3)
fig.savefig(fig_dir / "metrics_comparison_test.png", dpi=CFG["viz_dpi"])
plt.close(fig)


def overlay_liver_tumor(img_u8, liver_m, tumor_m, liver_rgb, tumor_rgb, a_liv=0.35, a_tum=0.42):
    rgb = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2RGB).astype(np.float32) / 255.0
    m_l = (liver_m > 0.5).astype(np.float32)[..., None]
    m_t = (tumor_m > 0.5).astype(np.float32)[..., None]
    col_l = np.zeros_like(rgb)
    col_l[..., 0], col_l[..., 1], col_l[..., 2] = liver_rgb
    col_t = np.zeros_like(rgb)
    col_t[..., 0], col_t[..., 1], col_t[..., 2] = tumor_rgb
    rgb = rgb * (1 - a_liv * m_l) + col_l * (a_liv * m_l)
    rgb = rgb * (1 - a_tum * m_t) + col_t * (a_tum * m_t)
    return np.clip(rgb * 255.0, 0, 255).astype(np.uint8)


@torch.no_grad()
def predict_batch(model, x):
    use_amp = CFG["device"] == "cuda"
    with torch.amp.autocast("cuda", enabled=use_amp):
        logits = model(x.to(CFG["device"]))
    return torch.sigmoid(logits).cpu()


def pick_studies(df, need_tumor, k, seed):
    rng = np.random.default_rng(seed)
    studies = df["study_number"].unique()
    pool = []
    for st in studies:
        st = int(st)
        sub = df[df["study_number"] == st]
        if need_tumor:
            if sub["tumor_mask_empty"].eq(False).any():
                pool.append(st)
        else:
            if sub["tumor_mask_empty"].eq(True).all():
                pool.append(st)
    pool = list(dict.fromkeys(pool))
    if len(pool) <= k:
        return pool
    return list(rng.choice(np.array(pool), size=k, replace=False))


def mid_index_for_study(df, study):
    idxs = np.flatnonzero((df["study_number"].values == study).astype(bool))
    return int(idxs[len(idxs) // 2])


models = {n: build_model_from_ckpt(p, CFG["device"])[0] for n, p in ckpt_paths.items()}
th_best = (detailed_summaries["best"]["th_liver"], detailed_summaries["best"]["th_tumor"])
th_latest = (detailed_summaries["latest"]["th_liver"], detailed_summaries["latest"]["th_tumor"])

studies_vis = []
studies_vis += pick_studies(test_df, True, CFG["viz_num_studies_with_tumor"], seed=11)
studies_vis += pick_studies(test_df, False, CFG["viz_num_studies_tumor_free"], seed=22)
studies_vis = list(dict.fromkeys(studies_vis))[
    : CFG["viz_num_studies_with_tumor"] + CFG["viz_num_studies_tumor_free"]
]

for st in studies_vis:
    j = mid_index_for_study(test_df, st)
    x, y, st_t = test_ds[j]
    assert int(st_t) == st
    xb = x.unsqueeze(0)
    pb_best = predict_batch(models["best"], xb)[0]
    pb_lat = predict_batch(models["latest"], xb)[0]

    def binarize(pb, thl, tht):
        return (pb[0] > thl).numpy(), (pb[1] > tht).numpy()

    gb_l, gb_t = binarize(pb_best, *th_best)
    gl_l, gl_t = binarize(pb_lat, *th_latest)

    img_u8 = (x.squeeze().numpy() * 255).astype(np.uint8)
    gt_l = y[0].numpy()
    gt_t = y[1].numpy()

    gt_blend = overlay_liver_tumor(img_u8, gt_l, gt_t, (0.0, 0.85, 0.2), (0.95, 0.15, 0.1))
    best_blend = overlay_liver_tumor(img_u8, gb_l, gb_t, (0.15, 0.55, 1.0), (1.0, 0.55, 0.05))
    latest_blend = overlay_liver_tumor(img_u8, gl_l, gl_t, (0.15, 0.55, 1.0), (1.0, 0.55, 0.05))

    err_best_t = np.abs(gb_t.astype(np.float32) - gt_t)
    err_lat_t = np.abs(gl_t.astype(np.float32) - gt_t)

    panels = [
        ("Input CT (preprocessed)", img_u8),
        ("GT liver+tumor", gt_blend),
        ("Best checkpoint", best_blend),
        ("Latest checkpoint", latest_blend),
        ("|Tumor err| best", (err_best_t * 255).astype(np.uint8)),
        ("|Tumor err| latest", (err_lat_t * 255).astype(np.uint8)),
    ]

    nrow, ncol = 2, 3
    fig, axs = plt.subplots(nrow, ncol, figsize=(ncol * 3.2, nrow * 3.4))
    for ax, (title, im) in zip(axs.ravel(), panels):
        if im.ndim == 2:
            ax.imshow(im, cmap="magma")
        else:
            ax.imshow(im)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    fig.suptitle(
        f"Study {st} | slice index {j} in test CSV | th best ({th_best[0]:.2f},{th_best[1]:.2f}) latest ({th_latest[0]:.2f},{th_latest[1]:.2f})",
        fontsize=10,
    )
    fig.savefig(fig_dir / f"qualitative_study_{st}.png", dpi=CFG["viz_dpi"])
    plt.close(fig)

print(f"Saved figures under: {fig_dir}")


## How to run

1. Attach **PNG volume dataset** + **val/test CSVs** + **model output** on Kaggle (or keep local paths).
2. Edit **Cell 1** overrides if auto-detection fails (`DATA_ROOT_OVERRIDE`, `VAL_CSV_OVERRIDE`, `TEST_CSV_OVERRIDE`, checkpoint paths).
3. **Run all cells**.

Artifacts:

| File | Purpose |
|------|---------|
| `results/checkpoint_comparison.csv` | Val/test metrics for both checkpoints |
| `results/per_study_test_dice.csv` | Per-patient mean slice Dice on test (for audits) |
| `results/checkpoint_recommendation.json` | Which checkpoint wins on locked test + thresholds |
| `results/checkpoint_eval_summary.json` | Compact per-checkpoint summary (no huge arrays) |
| `results/eval_analysis_log.txt` | Human-readable run log + supervised-learning notes + metric table |
| `results/eval_figures/metrics_comparison_test.png` | Bar chart |
| `results/eval_figures/qualitative_study_*.png` | GT vs best vs latest |

**Note:** Evaluation repeats full passes for the threshold grid; increase `threshold_grid_size` only when you need finer calibration.
